# Qwen MCQ: simplified KEFT-inspired alignment experiment

**This is a simplified experiment, not an exact KEFT reproduction.** Fresh 4-bit QLoRA adapters,
one epoch, the existing four balanced rotations, seed 42, learning rate 1e-4, batch 1,
gradient accumulation 16, rank 16, alpha 32, dropout 0.05, and 4,096-token student examples.

Reference: verbatim original Q/A found by `source_row`, then the rotated MCQ; frozen base
with LoRA disabled, `eval()` and `no_grad()`. Student: MCQ only, LoRA enabled.
Both vectors are the final normalized hidden state at their **last prompt token, before the answer**.
`total_loss = existing_LM_loss + 0.05 * mean(1 - cosine_similarity(student, reference))`.
Both terms use the existing accumulation-window normalization. References are computed on the fly;
there is no vector cache, second base-model copy, or extra student forward.

Attach a **private Kaggle dataset** containing every file from this bundle's `data` folder.
Enable **Internet** and **GPU**. The notebook uses one GPU. No API key is required for this public model.
All scripts and CPU tests are embedded. Start with a fresh session and use **Run All**.
For a background run choose **Save Version → Save & Run All**.

The default compares against the saved control (182/192 = 94.79% validation) without rerunning it.
You can enable a fresh control below for a stronger comparison. Test answers never enter references
or training. Final evaluation still includes fixed validation, original test, and rotated validation.
The base model/tokenizer revision is pinned to the saved control's commit before any training.


## Supervisor note

Alignment experiment inspired by KEFT. The implementation is simplified; results did not improve test accuracy, so alignment was disabled for the best continuation run.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
%pip install -q "transformers==4.57.3" "peft==0.17.1" "accelerate==1.15.0" "bitsandbytes==0.48.2" sentencepiece matplotlib


## Find and verify the attached data

Required filenames: `train.csv`, `validation.csv`, `TestData.csv`, `TrainData.csv`,
and `control_results.json`. `TrainData.csv` is the original QA, not the generated training MCQs.
Keep the bundled bytes unchanged; do not open/save the CSVs through Excel.
No laptop paths or upload widgets are used. If multiple matching datasets are attached, set `DATA_DIR` explicitly.


In [ ]:
import hashlib
import json
import shutil
import sys
from pathlib import Path
from datetime import datetime, timezone

WORKING_DIR = Path('/kaggle/working')
if not WORKING_DIR.is_dir():
    raise RuntimeError('Run this notebook on Kaggle with the data folder attached.')
os.chdir(WORKING_DIR)
DATA_DIR = None  # Optional explicit Path('/kaggle/input/<your-dataset>/data')
EXPECTED = {'train.csv': {'sha256': 'c2f759271d8a7d4f29105bfae4b523c31b73101a6042be1db5b5c3ddebc8a58f', 'rows': 1718}, 'validation.csv': {'sha256': 'c7c028971d4ba185113eefafbb74260db4e5484542c04fec96334e1b18c77e20', 'rows': 192}, 'TestData.csv': {'sha256': '5e10ec19d61624ee3554814337cf961fe499f4a666881c866c172fa1e51454a2', 'rows': 269}, 'TrainData.csv': {'sha256': '3150859588f37e4ec6b305ca8dce40726fed07b3a684ffbaaa70aec7c7dd61d3', 'rows': 1910}, 'control_results.json': {'sha256': '3d1776bcb534adcb133dcc8ec3f888dd3465a3a3d60c296f203bef4f977ea6cd'}}
if DATA_DIR is None:
    candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('TrainData.csv')
                         if all((p.parent / name).is_file() for name in EXPECTED)})
    if len(candidates) != 1:
        raise RuntimeError(f'Attach one dataset folder with all five files or set DATA_DIR; found {candidates}')
    DATA_DIR = candidates[0]
DATA_DIR = Path(DATA_DIR)
for name, expected in EXPECTED.items():
    source = DATA_DIR / name
    if hashlib.sha256(source.read_bytes()).hexdigest() != expected['sha256']:
        raise ValueError(f'{name} differs from the bundled input. Attach the unchanged bundle data; no integrity check is bypassed.')
    shutil.copyfile(source, WORKING_DIR / name)
print('Verified data:', DATA_DIR)


## Settings

`RUN_SMOKE = True` runs the actual GPU training/save/reload path on eight source questions,
including the longest eligible student and longest reference, before full training starts fresh.
The reference may be longer than 4,096 tokens: Q/A are never truncated and no extra groups are removed.
The reference limit is a fail-fast guard, not a training filter. If the GPU smoke runs out of memory,
keep the evidence and use a larger-memory GPU; do not treat the control's earlier smoke pass as validation of this new method.
Local full-data tokenizer preflight found a longest reference of 5,000 tokens (source 1839);
its student presentation is 4,005 tokens. GPU memory fit is still checked below.

`RUN_MATCHED_CONTROL = False` reuses the saved control. Set it to True to train another fresh
control under this notebook's environment and compare that instead. This adds another full training run.


In [ ]:
import torch
MAX_LENGTH = 4096
MAX_REFERENCE_LENGTH = 16384
SEED = 42
MODEL_REVISION = json.loads(Path('control_results.json').read_text())['metadata']['model_commit']
RUN_SMOKE = True
RUN_MATCHED_CONTROL = False
if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError('Enable a GPU and start a fresh session so CUDA_VISIBLE_DEVICES=0 takes effect.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
EXPERIMENT_DIR = WORKING_DIR / ('mcq-alignment-' + stamp)
EXPERIMENT_DIR.mkdir(exist_ok=False)
ALIGNMENT_DIR = EXPERIMENT_DIR / 'alignment'


## Write the embedded scripts and run CPU correctness tests

The tests use a tiny randomly initialized Qwen with real PEFT LoRA; they do not download or train the 7B model.


In [ ]:
# Embedded from benchmark.py
p = Path('benchmark.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Zero-shot MCQ baseline. No training or training-data access."""\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport re\nimport time\n\n\ndef build_prompt(row):\n    choices = "\\n".join(f"{key}) {row[key]}" for key in "abcd")\n    return (\n        "Answer the following multiple-choice question about a Unity C# game project.\\n"\n        "Choose the single best answer. End your response with Answer:a, Answer:b, "\n        "Answer:c, or Answer:d.\\n\\n"\n        f"{row[\'Question\']}\\n{choices}"\n    )\n\n\ndef parse_answer(response):\n    # An unfinished thinking block is not a final answer.\n    if "<think>" in response and "</think>" not in response:\n        return None\n    final = response.rsplit("</think>", 1)[-1].strip()\n    match = re.search(r"Answer\\s*:\\s*([abcd])\\s*[.!]?\\s*$", final, re.I)\n    if match:\n        return match.group(1).lower()\n    return final.lower() if re.fullmatch(r"[abcd]", final, re.I) else None\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", default="Qwen/Qwen3-4B")\n    parser.add_argument("--data", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--output", type=Path, required=True,\n                        help="New directory for this run; existing directories are rejected.")\n    parser.add_argument("--limit", type=int, help="First N rows for smoke checks only")\n    parser.add_argument("--thinking", action="store_true", help="Qwen3 thinking mode")\n    parser.add_argument("--max-new-tokens", type=int)\n    parser.add_argument("--four-bit", action="store_true")\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args(argv)\n    if args.limit is not None and args.limit < 1:\n        parser.error("--limit must be positive")\n    if args.max_new_tokens is not None and args.max_new_tokens < 1:\n        parser.error("--max-new-tokens must be positive")\n    if args.thinking and not args.model.startswith("Qwen/Qwen3-"):\n        parser.error("--thinking is implemented only for Qwen3 models")\n    with args.data.open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        required = {"class", "Question", "a", "b", "c", "d", "Answer"}\n        if not required.issubset(reader.fieldnames or []):\n            parser.error("Unexpected test CSV schema")\n        rows = list(reader)\n    if args.limit:\n        rows = rows[:args.limit]\n    if not rows or any(row["Answer"].strip().lower() not in "abcd" or\n                       len(row["Answer"].strip()) != 1 for row in rows):\n        parser.error("Empty dataset or invalid answer labels")\n\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("A CUDA GPU is required. Select a GPU runtime in Colab/Kaggle.")\n    args.output.mkdir(parents=True, exist_ok=False)\n    set_seed(args.seed)\n    tokenizer = AutoTokenizer.from_pretrained(args.model)\n    config = {"device_map": "auto", "torch_dtype": torch.float16}\n    if args.four_bit:\n        config["quantization_config"] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,\n        )\n    model = AutoModelForCausalLM.from_pretrained(args.model, **config).eval()\n    max_tokens = args.max_new_tokens or (2048 if args.thinking else 128)\n    generation = {"max_new_tokens": max_tokens, "do_sample": args.thinking,\n                  "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id}\n    if args.thinking:\n        generation.update(temperature=0.6, top_p=0.95, top_k=20)\n    metadata = {\n        "model": args.model, "model_commit": getattr(model.config, "_commit_hash", None),\n        "dataset_sha256": hashlib.sha256(args.data.read_bytes()).hexdigest(),\n        "rows": len(rows), "smoke_test": args.limit is not None, "seed": args.seed,\n        "thinking": args.thinking, "four_bit": args.four_bit, "generation": generation,\n        "gpu": torch.cuda.get_device_name(0), "prompt_example": build_prompt(rows[0]),\n        "versions": {p: importlib.metadata.version(p) for p in\n                     ["torch", "transformers", "accelerate", "bitsandbytes"]},\n        "protocol": "Zero-shot; no training examples or source code; invalid answers count as wrong.",\n    }\n    (args.output / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\n    predictions = []\n    start = time.perf_counter()\n    with (args.output / "predictions.jsonl").open("x", encoding="utf-8") as handle:\n        for index, row in enumerate(rows):\n            messages = [{"role": "user", "content": build_prompt(row)}]\n            template_args = {}\n            if args.model.startswith("Qwen/Qwen3-"):\n                template_args["enable_thinking"] = args.thinking\n            text = tokenizer.apply_chat_template(\n                messages, tokenize=False, add_generation_prompt=True, **template_args)\n            inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)\n            tick = time.perf_counter()\n            with torch.inference_mode():\n                output = model.generate(**inputs, **generation)\n            generated = output[0, inputs.input_ids.shape[1]:]\n            response = tokenizer.decode(generated, skip_special_tokens=True)\n            answer = parse_answer(response)\n            result = {\n                "row": index, "class": row["class"], "question": row["Question"],\n                "response": response, "prediction": answer,\n                "truth": row["Answer"].strip().lower(),\n                "correct": answer == row["Answer"].strip().lower(),\n                "seconds": time.perf_counter() - tick,\n                "generated_tokens": len(generated),\n                "token_budget_reached": len(generated) >= max_tokens,\n            }\n            predictions.append(result)\n            handle.write(json.dumps(result, ensure_ascii=False) + "\\n")\n            handle.flush()\n            print(f"{index + 1}/{len(rows)} correct={sum(p[\'correct\'] for p in predictions)}", flush=True)\n    correct = sum(p["correct"] for p in predictions)\n    labels = {key: sum(p["truth"] == key for p in predictions) for key in "abcd"}\n    summary = {\n        "model": args.model, "questions": len(rows), "correct": correct,\n        "accuracy": correct / len(rows),\n        "invalid_answers": sum(p["prediction"] is None for p in predictions),\n        "token_budget_reached": sum(p["token_budget_reached"] for p in predictions),\n        "uniform_random_expected_accuracy": 0.25,\n        "label_counts": labels, "constant_choice_accuracy": {k: v / len(rows) for k, v in labels.items()},\n        "elapsed_seconds": time.perf_counter() - start,\n        "smoke_test": args.limit is not None,\n    }\n    (args.output / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
# Embedded from finetune_qwen_coder.py
p = Path('finetune_qwen_coder.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""One-epoch QLoRA experiment; test labels never enter training or prompts."""\n\nimport argparse\nfrom collections import defaultdict\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport time\n\nos.environ.setdefault("USE_TF", "0")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nfrom benchmark import build_prompt, parse_answer\n\nMODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n\n\ndef read_csv(path, required):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not set(required).issubset(reader.fieldnames or []):\n            raise ValueError(f"Unexpected CSV schema: {path}")\n        rows = list(reader)\n    if not rows or any(not str(row.get(key) or "").strip() for row in rows for key in required):\n        raise ValueError(f"Empty rows or required fields: {path}")\n    return rows\n\n\ndef split_training_rows(rows, seed=42, validation_fraction=0.1):\n    # Keep identical questions together even if class names or answers differ.\n    groups = defaultdict(list)\n    for index, row in enumerate(rows):\n        question = re.sub(r"\\s+", " ", row["Q"].strip().casefold())\n        groups[question].append(index)\n    keys = sorted(groups)\n    random.Random(seed).shuffle(keys)\n    validation_keys = set(keys[:max(1, round(len(keys) * validation_fraction))])\n    validation_indices = sorted(i for key in validation_keys for i in groups[key])\n    validation_set = set(validation_indices)\n    training_indices = [i for i in range(len(rows)) if i not in validation_set]\n    if not training_indices or not validation_indices:\n        raise ValueError("Need enough distinct questions for train/validation splitting")\n    return training_indices, validation_indices\n\n\ndef tokenize_example(row, tokenizer, max_length):\n    user = f"In this Unity game project, regarding {row[\'class\'].strip()}:\\n{row[\'Q\'].strip()}"\n    messages = [{"role": "user", "content": user}]\n    prefix = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)\n    full = tokenizer.apply_chat_template(\n        messages + [{"role": "assistant", "content": row["A"].strip()}],\n        tokenize=True, add_generation_prompt=False,\n    )\n    if full[:len(prefix)] != prefix:\n        raise ValueError("Chat-template prefix mismatch; refusing incorrect loss masking")\n    if len(full) > max_length:\n        return None, len(full)\n    labels = [-100] * len(prefix) + full[len(prefix):]\n    if not any(label != -100 for label in labels):\n        raise ValueError("Example contains no answer tokens")\n    return {"input_ids": full, "attention_mask": [1] * len(full), "labels": labels}, len(full)\n\n\ndef collate_examples(examples, pad_token_id):\n    import torch\n    width = math.ceil(max(len(row["input_ids"]) for row in examples) / 8) * 8\n    batch = {key: [] for key in ["input_ids", "attention_mask", "labels"]}\n    for row in examples:\n        padding = width - len(row["input_ids"])\n        for key, pad in [("input_ids", pad_token_id), ("attention_mask", 0), ("labels", -100)]:\n            batch[key].append(row[key] + [pad] * padding)\n    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}\n\n\ndef evaluate_mcq(model, tokenizer, rows, output):\n    import torch\n    output.mkdir(parents=True, exist_ok=False)\n    model.eval()\n    model.config.use_cache = True\n    predictions = []\n    torch.cuda.synchronize()\n    start = time.perf_counter()\n    with (output / "predictions.jsonl").open("x", encoding="utf-8") as handle:\n        for index, row in enumerate(rows):\n            prompt = build_prompt(row)\n            text = tokenizer.apply_chat_template(\n                [{"role": "user", "content": prompt}],\n                tokenize=False, add_generation_prompt=True,\n            )\n            inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt").to(model.device)\n            tick = time.perf_counter()\n            with torch.inference_mode():\n                generated = model.generate(\n                    **inputs, do_sample=False, max_new_tokens=128,\n                    pad_token_id=tokenizer.pad_token_id,\n                )[0, inputs.input_ids.shape[1]:]\n            response = tokenizer.decode(generated, skip_special_tokens=True)\n            answer = parse_answer(response)\n            result = {\n                "row": index, "class": row["class"], "question": row["Question"],\n                "response": response, "prediction": answer, "truth": row["Answer"].strip().lower(),\n                "correct": answer == row["Answer"].strip().lower(),\n                "generated_tokens": len(generated), "token_budget_reached": len(generated) >= 128,\n                "seconds": time.perf_counter() - tick,\n            }\n            predictions.append(result)\n            handle.write(json.dumps(result, ensure_ascii=False) + "\\n")\n            handle.flush()\n            if (index + 1) % 10 == 0 or index + 1 == len(rows):\n                print(f"{output.name}: {index + 1}/{len(rows)}", flush=True)\n    summary = {\n        "model": MODEL_ID, "questions": len(rows),\n        "correct": sum(row["correct"] for row in predictions),\n        "accuracy": sum(row["correct"] for row in predictions) / len(rows),\n        "invalid_answers": sum(row["prediction"] is None for row in predictions),\n        "token_budget_reached": sum(row["token_budget_reached"] for row in predictions),\n        "elapsed_seconds": time.perf_counter() - start,\n    }\n    write_json(output / "summary.json", summary)\n    with (output / "predictions.csv").open("x", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(predictions[0]))\n        writer.writeheader()\n        writer.writerows(predictions)\n    return summary, predictions\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train", type=Path, default=Path("TrainData.csv"))\n    parser.add_argument("--test", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--max-length", type=int, default=2048)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    if args.max_length < 64:\n        parser.error("--max-length must be at least 64")\n    train_rows = read_csv(args.train, ["class", "Q", "A"])\n    test_rows = read_csv(args.test, ["class", "Question", "a", "b", "c", "d", "Answer"])\n    if any(row["Answer"].strip().lower() not in list("abcd") for row in test_rows):\n        raise ValueError("Invalid MCQ answer label")\n\n    import torch\n    from functools import partial\n    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a CUDA GPU before running this experiment")\n    args.output.mkdir(parents=True, exist_ok=False)\n    set_seed(args.seed)\n    training_indices, validation_indices = split_training_rows(train_rows, args.seed)\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    prepared = {}\n    lengths = []\n    exclusions = []\n    for index, row in enumerate(train_rows):\n        example, length = tokenize_example(row, tokenizer, args.max_length)\n        lengths.append(length)\n        if example is None:\n            exclusions.append({"row": index, "class": row["class"], "tokens": length,\n                               "reason": "Exceeds max_length; excluded without truncating answers"})\n        else:\n            prepared[index] = example\n    training_indices = [i for i in training_indices if i in prepared]\n    validation_indices = [i for i in validation_indices if i in prepared]\n    if not training_indices or not validation_indices:\n        raise ValueError("No usable training or validation examples after length checks")\n    write_json(args.output / "split.json", {\n        "seed": args.seed, "train_indices": training_indices,\n        "validation_indices": validation_indices, "excluded": exclusions,\n        "note": "Grouped by identical normalized question. Semantic paraphrases may still overlap.",\n    })\n    quantization = BitsAndBytesConfig(\n        load_in_4bit=True, bnb_4bit_quant_type="nf4",\n        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,\n    )\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL_ID, quantization_config=quantization, dtype=torch.float16,\n        device_map={"": 0}, attn_implementation="sdpa",\n    )\n    revision = getattr(model.config, "_commit_hash", None)\n    training_settings = {\n        "num_train_epochs": 1, "learning_rate": 1e-4, "batch_size": 1,\n        "gradient_accumulation_steps": 16, "warmup_ratio": 0.05,\n        "lr_scheduler_type": "cosine", "weight_decay": 0.01,\n        "max_grad_norm": 0.3, "lora_r": 16, "lora_alpha": 32,\n        "lora_dropout": 0.05, "max_length": args.max_length,\n    }\n    metadata = {\n        "model": MODEL_ID, "model_commit": revision, "seed": args.seed,\n        "gpu": torch.cuda.get_device_name(0), "gpu_count_used": 1,\n        "train_sha256": hashlib.sha256(args.train.read_bytes()).hexdigest(),\n        "test_sha256": hashlib.sha256(args.test.read_bytes()).hexdigest(),\n        "train_rows": len(train_rows), "train_examples_used": len(training_indices),\n        "validation_examples": len(validation_indices), "test_rows": len(test_rows),\n        "excluded_long_examples": len(exclusions),\n        "train_token_lengths": {"min": min(lengths), "max": max(lengths), "mean": sum(lengths)/len(lengths)},\n        "settings": training_settings,\n        "versions": {p: importlib.metadata.version(p) for p in\n                     ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]},\n        "protocol": "One epoch on training split only. Answer-only loss. Test evaluated once before and once after training, never used for selection. Fresh base and adapter share decoding and packages.",\n        "evaluation": {"max_new_tokens": 128, "do_sample": False,\n                       "invalid_answers_count_as_wrong": True, "prompt_example": build_prompt(test_rows[0])},\n    }\n    write_json(args.output / "metadata.json", metadata)\n    print(f"Training: {len(training_indices)}; validation: {len(validation_indices)}; exclusions: {len(exclusions)}", flush=True)\n    print("Measuring a fresh base-model reference with the same environment...", flush=True)\n    base_summary, base_predictions = evaluate_mcq(model, tokenizer, test_rows, args.output / "base")\n\n    # Reset stochastic state so training does not depend on evaluation RNG consumption.\n    set_seed(args.seed)\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    model = get_peft_model(model, LoraConfig(\n        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",\n        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],\n    ))\n    model.print_trainable_parameters()\n    trainable, total = model.get_nb_trainable_parameters()\n    metadata.update(trainable_parameters=trainable, total_parameters=total)\n    write_json(args.output / "metadata.json", metadata)\n    training_args = TrainingArguments(\n        output_dir=str(args.output / "training"), num_train_epochs=1,\n        learning_rate=1e-4, per_device_train_batch_size=1, per_device_eval_batch_size=1,\n        gradient_accumulation_steps=16, gradient_checkpointing=True,\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n        fp16=True, optim="paged_adamw_8bit", lr_scheduler_type="cosine",\n        warmup_ratio=0.05, weight_decay=0.01, max_grad_norm=0.3,\n        logging_steps=5, eval_strategy="epoch", save_strategy="no",\n        prediction_loss_only=True, report_to="none", seed=args.seed, data_seed=args.seed,\n        dataloader_num_workers=0, remove_unused_columns=False,\n    )\n    trainer = Trainer(\n        model=model, args=training_args,\n        train_dataset=[prepared[i] for i in training_indices],\n        eval_dataset=[prepared[i] for i in validation_indices],\n        processing_class=tokenizer,\n        data_collator=partial(collate_examples, pad_token_id=tokenizer.pad_token_id),\n    )\n    torch.cuda.reset_peak_memory_stats()\n    trained = trainer.train()\n    write_json(args.output / "training_metrics.json", {\n        **trained.metrics, "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated()/1024**3,\n    })\n    write_json(args.output / "training_log.json", trainer.state.log_history)\n    adapter_path = args.output / "adapter"\n    model.save_pretrained(adapter_path, safe_serialization=True)\n    tokenizer.save_pretrained(adapter_path)\n\n    # Reload from disk before final scoring, verifying adapter portability.\n    del trainer, model\n    gc.collect()\n    torch.cuda.empty_cache()\n    base = AutoModelForCausalLM.from_pretrained(\n        MODEL_ID, revision=revision, quantization_config=quantization,\n        dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa",\n    )\n    model = PeftModel.from_pretrained(base, str(adapter_path), is_trainable=False)\n    tuned_summary, tuned_predictions = evaluate_mcq(model, tokenizer, test_rows, args.output / "finetuned")\n    comparison = {\n        "base": base_summary, "finetuned": tuned_summary,\n        "accuracy_change_percentage_points": 100*(tuned_summary["accuracy"]-base_summary["accuracy"]),\n        "relative_accuracy_change": (tuned_summary["accuracy"]-base_summary["accuracy"])/base_summary["accuracy"] if base_summary["accuracy"] else None,\n        "wrong_to_correct": sum(not a["correct"] and b["correct"] for a,b in zip(base_predictions,tuned_predictions)),\n        "correct_to_wrong": sum(a["correct"] and not b["correct"] for a,b in zip(base_predictions,tuned_predictions)),\n        "saved_adapter_reload_verified": True,\n    }\n    write_json(args.output / "comparison.json", comparison)\n    print(json.dumps(comparison, indent=2), flush=True)\n    print(f"Completed. Adapter and results: {args.output}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
# Embedded from finetune_qwen_mcq.py
p = Path('finetune_qwen_mcq.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Four-exposure MCQ QLoRA experiment; original assessment evaluator is unchanged."""\n\nimport argparse\nfrom collections import Counter\nfrom functools import partial\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport re\n\n# Set before torch initializes in the training subprocess.\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nfrom benchmark import build_prompt\nfrom finetune_qwen_coder import MODEL_ID, collate_examples, evaluate_mcq, read_csv, write_json\n\nLETTERS = "abcd"\nMCQ_FIELDS = ["class", "Question", *LETTERS, "Answer"]\n\n\ndef assistant_only_loss(model, inputs, return_outputs=False, num_items_in_batch=None):\n    """Exact masked causal loss without materializing prompt vocabulary logits.\n\n    Qwen\'s logits_to_keep slices hidden states before the vocabulary projection.\n    Keep the predecessor of each supervised label, including the turn ending.\n    This also works with padding and different answer positions within a batch.\n    """\n    import torch\n    labels = inputs["labels"]\n    targets = labels[:, 1:]\n    positions = torch.where((targets != -100).any(dim=0))[0]\n    if positions.numel() == 0:\n        raise ValueError("Batch has no supervised next-token targets")\n    forward_inputs = {k: v for k, v in inputs.items() if k != "labels"}\n    outputs = model(**forward_inputs, logits_to_keep=positions, use_cache=False)\n    selected_targets = targets.index_select(1, positions).to(outputs.logits.device)\n    loss = torch.nn.functional.cross_entropy(\n        outputs.logits.float().reshape(-1, outputs.logits.shape[-1]),\n        selected_targets.reshape(-1), ignore_index=-100,\n        reduction="sum" if num_items_in_batch is not None else "mean",\n    )\n    if num_items_in_batch is not None:\n        # Preserve Trainer\'s token normalization across gradient accumulation.\n        if isinstance(num_items_in_batch, torch.Tensor):\n            num_items_in_batch = num_items_in_batch.to(loss.device)\n        loss = loss / num_items_in_batch\n    return (loss, outputs) if return_outputs else loss\n\n\ndef load_split(path, split):\n    rows = read_csv(path, MCQ_FIELDS + ["source_row", "split"])\n    ids = set()\n    for row in rows:\n        if row["split"] != split:\n            raise ValueError(f"Wrong split in {path}")\n        try:\n            source_id = int(row["source_row"])\n        except ValueError as exc:\n            raise ValueError("Invalid source_row") from exc\n        if source_id < 0 or source_id in ids:\n            raise ValueError("Invalid or repeated source_row within a split")\n        ids.add(source_id)\n        row["Answer"] = row["Answer"].strip().lower()\n        if row["Answer"] not in list(LETTERS):\n            raise ValueError("Invalid answer letter")\n        if len({row[letter].strip() for letter in LETTERS}) != 4:\n            raise ValueError("Duplicate MCQ options")\n    return rows\n\n\ndef check_split_isolation(train, validation):\n    if {r["source_row"] for r in train} & {r["source_row"] for r in validation}:\n        raise ValueError("Source ids overlap between train and validation")\n    key = lambda r: re.sub(r"\\s+", " ", r["Question"].strip()).casefold()\n    if {key(r) for r in train} & {key(r) for r in validation}:\n        raise ValueError("Identical normalized questions cross splits")\n\n\ndef rotate_question(row, seed=42):\n    """Shuffle once per stable source id, then rotate all four options together."""\n    order = list(LETTERS)\n    seed_bytes = f"mcq-rotation-v1:{seed}:{row[\'source_row\']}".encode()\n    random.Random(int.from_bytes(hashlib.sha256(seed_bytes).digest(), "big")).shuffle(order)\n    result = []\n    for rotation in range(4):\n        permutation = order[rotation:] + order[:rotation]\n        mapping = dict(zip(LETTERS, permutation))  # new letter -> original letter\n        answer = next(new for new, old in mapping.items() if old == row["Answer"])\n        rotated = {**row, **{new: row[old] for new, old in mapping.items()}, "Answer": answer}\n        result.append((rotated, {"source_row": int(row["source_row"]), "split": row["split"],\n                                  "rotation": rotation, "new_to_original": mapping,\n                                  "original_answer": row["Answer"], "answer": answer}))\n    return result\n\n\ndef tokenize_mcq(row, tokenizer, max_length):\n    messages = [{"role": "user", "content": build_prompt(row)}]\n    prefix = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=False)\n    full = tokenizer.apply_chat_template(\n        messages + [{"role": "assistant", "content": "Answer:" + row["Answer"]}],\n        tokenize=True, add_generation_prompt=False, return_dict=False,\n    )\n    if full[:len(prefix)] != prefix:\n        raise ValueError("Chat-template prefix mismatch; refusing incorrect loss masking")\n    if len(full) <= len(prefix):\n        raise ValueError("No assistant target tokens")\n    if len(full) > max_length:\n        return None, len(full)\n    return {"input_ids": full, "attention_mask": [1] * len(full),\n            "labels": [-100] * len(prefix) + full[len(prefix):]}, len(full)\n\n\ndef prepare_split(rows, tokenizer, max_length, seed, training):\n    prepared, admitted, maps, exclusions = [], [], [], []\n    for row in rows:\n        variants = rotate_question(row, seed)\n        encoded = [tokenize_mcq(v, tokenizer, max_length) for v, _ in variants]\n        fixed, fixed_length = tokenize_mcq(row, tokenizer, max_length)\n        lengths = [length for _, length in encoded]\n        # Use the same length-eligible validation cohort for fixed and rotated metrics.\n        if any(example is None for example, _ in encoded) or fixed is None:\n            exclusions.append({"source_row": int(row["source_row"]), "split": row["split"],\n                               "rotation_lengths": lengths, "fixed_length": fixed_length,\n                               "reason": "Entire question group exceeds max_length; no truncation"})\n            continue\n        admitted.append(row)\n        if training:\n            prepared.extend(example for example, _ in encoded)\n        else:\n            prepared.append(fixed)\n        for (_, mapping), length in zip(variants, lengths):\n            maps.append({**mapping, "tokens": length})\n    if not admitted:\n        raise ValueError("No questions fit max_length in a split")\n    if training:\n        random.Random(seed).shuffle(prepared)\n        assert Counter(m["answer"] for m in maps) == {l: len(admitted) for l in LETTERS}\n    return prepared, admitted, maps, exclusions\n\n\ndef smoke_training_rows(rows, tokenizer, max_length, seed):\n    """Include the longest fitting question so smoke tests exercise peak lengths."""\n    _, admitted, maps, _ = prepare_split(rows, tokenizer, max_length, seed, False)\n    longest_id = max(maps, key=lambda m: m["tokens"])["source_row"]\n    longest = next(r for r in admitted if int(r["source_row"]) == longest_id)\n    return [longest] + [r for r in admitted if r is not longest][:7]\n\n\ndef rotation_diagnostics(rows, predictions, seed):\n    if len(predictions) != 4 * len(rows):\n        raise ValueError("Incomplete rotated predictions")\n    details, by_position = [], {l: {"correct": 0, "questions": 0} for l in LETTERS}\n    for index, row in enumerate(rows):\n        selected = []\n        correct = []\n        for offset, (_, mapping) in enumerate(rotate_question(row, seed)):\n            prediction = predictions[4 * index + offset]\n            letter = prediction["prediction"]\n            selected.append(mapping["new_to_original"].get(letter))\n            correct.append(prediction["correct"])\n            stat = by_position[mapping["answer"]]\n            stat["questions"] += 1\n            stat["correct"] += prediction["correct"]\n        details.append({"source_row": int(row["source_row"]),\n                        "selected_original_letters": selected,\n                        "consistent": None not in selected and len(set(selected)) == 1,\n                        "all_four_correct": all(correct), "correct_rotations": sum(correct)})\n    for stat in by_position.values():\n        stat["accuracy"] = stat["correct"] / stat["questions"]\n    return {"questions": len(rows), "accuracy_by_correct_position": by_position,\n            "consistent_answer_text_rate": sum(r["consistent"] for r in details) / len(rows),\n            "all_four_correct_rate": sum(r["all_four_correct"] for r in details) / len(rows),\n            "note": "Invalid responses are inconsistent. Consistency alone does not imply correctness.",\n            "per_question": details}\n\n\ndef score(model, tokenizer, validation, test, output, seed):\n    # Do not build a graph or retain KV caching during loss evaluation in Trainer.\n    fixed, _ = evaluate_mcq(model, tokenizer, validation, output / "validation")\n    test_summary, test_predictions = evaluate_mcq(model, tokenizer, test, output / "test")\n    rotated = [v for row in validation for v, _ in rotate_question(row, seed)]\n    rotated_summary, predictions = evaluate_mcq(model, tokenizer, rotated, output / "validation_rotated")\n    diagnostic = rotation_diagnostics(validation, predictions, seed)\n    diagnostic["aggregate"] = rotated_summary\n    write_json(output / "rotation_diagnostics.json", diagnostic)\n    return {"validation": fixed, "test": test_summary, "rotation": diagnostic}, test_predictions\n\n\ndef make_comparison(base_summary, base_predictions, tuned_summary, tuned_predictions, smoke_test=False):\n    result = {"base": base_summary, "finetuned": tuned_summary, "smoke_test": smoke_test,\n              "baseline_evaluation": "skipped" if base_summary is None else "measured_this_run",\n              "test_accuracy_change_percentage_points": None,\n              "wrong_to_correct": None, "correct_to_wrong": None,\n              "saved_adapter_reload_verified": True}\n    if base_summary is not None:\n        result.update(\n            test_accuracy_change_percentage_points=100 * (tuned_summary["test"]["accuracy"] - base_summary["test"]["accuracy"]),\n            wrong_to_correct=sum(not a["correct"] and b["correct"] for a, b in zip(base_predictions, tuned_predictions)),\n            correct_to_wrong=sum(a["correct"] and not b["correct"] for a, b in zip(base_predictions, tuned_predictions)),\n        )\n    return result\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train", type=Path, default=Path("data/mcq/generated-mcqs-final/train.csv"))\n    parser.add_argument("--validation", type=Path, default=Path("data/mcq/generated-mcqs-final/validation.csv"))\n    parser.add_argument("--test", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--max-length", type=int, default=4096)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--revision", help="Optional base/tokenizer commit pin for a matched experiment")\n    parser.add_argument("--prepare-only", action="store_true", help="Tokenizer preflight; no model load or GPU required")\n    parser.add_argument("--smoke-test", action="store_true", help="8 train questions, 4 validation, 2 test; not benchmark results")\n    parser.add_argument("--skip-base-evaluation", action="store_true", help="Train immediately; score only the saved/reloaded fine-tuned adapter")\n    parser.add_argument("--alignment-qa", type=Path, help="Enable simplified alignment using verbatim original class,Q,A CSV; omit for unchanged control")\n    parser.add_argument("--max-reference-length", type=int, default=16384, help="Fail above this reference length; never truncate QA or exclude extra groups")\n    args = parser.parse_args(argv)\n    if args.max_length < 64:\n        parser.error("max-length must be at least 64")\n    train = load_split(args.train, "train")\n    validation = load_split(args.validation, "validation")\n    check_split_isolation(train, validation)\n    total_train, total_validation = len(train), len(validation)\n    args.output.mkdir(parents=True, exist_ok=False)\n    from transformers import AutoTokenizer, set_seed\n    set_seed(args.seed)\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=args.revision)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    original_qa = None\n    if args.alignment_qa:\n        from mcq_alignment import load_original_qa\n        original_qa = load_original_qa(args.alignment_qa)\n    if args.smoke_test:\n        if original_qa is None:\n            train = smoke_training_rows(train, tokenizer, args.max_length, args.seed)\n        else:\n            from mcq_alignment import alignment_smoke_rows\n            train = alignment_smoke_rows(train, original_qa, tokenizer, args.max_length, args.max_reference_length, args.seed)\n        validation = validation[:4]\n    train_data, train, train_maps, train_exclusions = prepare_split(train, tokenizer, args.max_length, args.seed, True)\n    val_data, validation, val_maps, val_exclusions = prepare_split(validation, tokenizer, args.max_length, args.seed, False)\n    alignment_lengths = None\n    if original_qa is not None:\n        from mcq_alignment import attach_references\n        alignment_lengths = attach_references(train_data, train, original_qa, tokenizer, args.max_reference_length, args.seed)\n        write_json(args.output / "alignment_lengths.json", alignment_lengths)\n    metadata = {\n        "model": MODEL_ID, "seed": args.seed, "smoke_test": args.smoke_test,\n        "source_counts": {"train": total_train, "validation": total_validation},\n        "admitted_questions": {"train": len(train), "validation": len(validation)},\n        "training_presentations": len(train_data), "train_label_counts": dict(Counter(m["answer"] for m in train_maps)),\n        "training_source_rows": [int(r["source_row"]) for r in train],\n        "input_sha256": {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in\n                         [("train", args.train), ("validation", args.validation)]},\n        "baseline_evaluation": "skipped" if args.skip_base_evaluation else "measure_this_run",\n        "protocol": "One epoch of four rotations per question (four original-question exposures). Fixed validation plus rotation diagnostics after adapter reload. Baseline scoring is optional; skipped baselines are never fabricated or imported from unverified prior runs. Test labels never enter training. Not the original single-exposure assessment budget.",\n        "settings": {"rotations": 4, "max_length": args.max_length, "num_train_epochs": 1,\n                     "learning_rate": 1e-4, "batch_size": 1, "gradient_accumulation_steps": 16,\n                     "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,\n                     "target_format": "Answer:<letter>", "loss": "assistant response only",\n                     "loss_implementation": "selected causal logits before vocabulary projection",\n                     "allocator": os.environ.get("PYTORCH_CUDA_ALLOC_CONF")},\n        "evaluation": {"max_new_tokens": 128, "do_sample": False, "invalid_answers_count_as_wrong": True,\n                       "prompt_example": build_prompt(validation[0]),\n                       "validation_cohort": "All four orders and original order must fit max_length; full exclusions in length_exclusions.json"},\n    }\n    write_json(args.output / "metadata.json", metadata)\n    if original_qa is not None:\n        metadata["alignment"] = {\n            "weight": 0.05, "description": "Simplified QA-conditioned cosine alignment; not an exact KEFT reproduction",\n            "original_qa_sha256": hashlib.sha256(args.alignment_qa.read_bytes()).hexdigest(),\n            "original_qa_rows": len(original_qa), "qa_preservation": "Verbatim parsed Q and A; no stripping or truncation",\n            "reference": "Original QA then rotated MCQ; LoRA disabled; eval; no_grad; decoder final normalized hidden vector at last prompt token",\n            "student": "MCQ only; LoRA enabled; same last-prompt-token position before Answer:<letter>",\n            "reference_vectors": "Computed on the fly, before each student forward; never cached",\n            "max_reference_length": args.max_reference_length,\n            "max_observed_reference_tokens": max(x["reference_tokens"] for x in alignment_lengths),\n            "validation": "MCQ only, unchanged LM loss and generation evaluation; no validation QA reference",\n        }\n        write_json(args.output / "metadata.json", metadata)\n    write_json(args.output / "length_exclusions.json", train_exclusions + val_exclusions)\n    with (args.output / "rotation_map.jsonl").open("w", encoding="utf-8") as handle:\n        for mapping in train_maps + val_maps:\n            handle.write(json.dumps(mapping) + "\\n")\n    print(f"Admitted: {len(train)} train questions -> {len(train_data)} presentations; {len(validation)} validation questions. Excluded groups: {len(train_exclusions) + len(val_exclusions)}.", flush=True)\n    if args.prepare_only:\n        print("Preflight complete; no model was loaded or trained.", flush=True)\n        return\n\n    import torch\n    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Colab CUDA GPU. Tokenizer preflight is saved.")\n    test = read_csv(args.test, MCQ_FIELDS)\n    if any(r["Answer"].strip().lower() not in list(LETTERS) for r in test):\n        raise ValueError("Invalid test answer label")\n    if args.smoke_test:\n        test = test[:2]\n    # Include the assistant target in context checks; never truncate evaluation silently.\n    if any(tokenize_mcq(r, tokenizer, args.max_length)[0] is None for r in test):\n        raise ValueError("A test prompt exceeds max_length; increase the configured context before running")\n    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                                   bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)\n    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=args.revision, quantization_config=quantization,\n                                                dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")\n    revision = getattr(model.config, "_commit_hash", None)\n    if alignment_lengths and max(x["reference_tokens"] for x in alignment_lengths) > model.config.max_position_embeddings:\n        raise ValueError("A reference exceeds the model context window. No truncation or extra exclusions allowed.")\n    metadata.update(model_commit=revision, gpu=torch.cuda.get_device_name(0), gpu_count_used=1,\n                    test_rows=len(test), test_sha256=hashlib.sha256(args.test.read_bytes()).hexdigest(),\n                    versions={p: importlib.metadata.version(p) for p in\n                              ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]})\n    write_json(args.output / "metadata.json", metadata)\n    base_summary, base_predictions = None, None\n    if args.skip_base_evaluation:\n        print("Skipping baseline evaluation. Starting fine-tuning from fresh base weights.", flush=True)\n    else:\n        base_summary, base_predictions = score(model, tokenizer, validation, test, args.output / "base", args.seed)\n    set_seed(args.seed)\n    model.train()\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,\n        bias="none", task_type="CAUSAL_LM",\n        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))\n    model.print_trainable_parameters()\n    trainable, total = model.get_nb_trainable_parameters()\n    metadata.update(trainable_parameters=trainable, total_parameters=total)\n    write_json(args.output / "metadata.json", metadata)\n    training_args = TrainingArguments(\n        output_dir=str(args.output / "training"), num_train_epochs=1, learning_rate=1e-4,\n        per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=16,\n        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},\n        fp16=True, optim="paged_adamw_8bit", lr_scheduler_type="cosine", warmup_ratio=0.05,\n        weight_decay=0.01, max_grad_norm=0.3, logging_steps=1 if args.smoke_test else 5,\n        eval_strategy="epoch", save_strategy="no", prediction_loss_only=True, report_to="none",\n        seed=args.seed, data_seed=args.seed, dataloader_num_workers=0, remove_unused_columns=False,\n    )\n    objective = None\n    collator = partial(collate_examples, pad_token_id=tokenizer.pad_token_id)\n    callbacks = []\n    if original_qa is not None:\n        from mcq_alignment import AlignmentObjective, collate_alignment\n        from transformers import TrainerCallback\n        objective = AlignmentObjective(0.05)\n        collator = partial(collate_alignment, pad_token_id=tokenizer.pad_token_id)\n\n        class GradientAudit(TrainerCallback):\n            checked = False\n\n            def on_pre_optimizer_step(self, args_, state, control, model=None, **kwargs):\n                if self.checked:\n                    return\n                grads = [p.grad for n, p in model.named_parameters() if "lora_" in n and p.grad is not None]\n                if not grads or not all(torch.isfinite(g).all() for g in grads) or not any(g.abs().max() > 0 for g in grads):\n                    raise FloatingPointError("LoRA gradients must be present, finite, and nonzero")\n                if any(p.grad is not None for n, p in model.named_parameters() if "lora_" not in n):\n                    raise RuntimeError("Frozen base parameters received gradients")\n                write_json(args.output / "alignment_gradient_check.json", {\n                    "lora_gradients_present_finite_nonzero": True,\n                    "frozen_base_has_no_gradients": True,\n                    "reference_has_no_gradients": all(not x["reference_requires_grad"] and x["reference_grad_fn_is_none"] for x in objective.log),\n                    "optimizer_step_before_update": state.global_step,\n                })\n                self.checked = True\n\n        callbacks = [GradientAudit()]\n\n    class MCQTrainer(Trainer):\n        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n            if objective is not None:\n                return objective(model, inputs, return_outputs, num_items_in_batch)\n            return assistant_only_loss(model, inputs, return_outputs, num_items_in_batch)\n\n    trainer = MCQTrainer(model=model, args=training_args, train_dataset=train_data, eval_dataset=val_data,\n                      processing_class=tokenizer, data_collator=collator, callbacks=callbacks)\n    trainer.model_accepts_loss_kwargs = True\n    gc.collect()\n    torch.cuda.empty_cache()\n    torch.cuda.reset_peak_memory_stats()\n    try:\n        trained = trainer.train()\n    finally:\n        if objective is not None:\n            write_json(args.output / "alignment_loss_log.json", objective.log)\n    write_json(args.output / "training_metrics.json", {**trained.metrics,\n               "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3})\n    write_json(args.output / "training_log.json", trainer.state.log_history)\n    adapter = args.output / "adapter"\n    model.save_pretrained(adapter, safe_serialization=True)\n    tokenizer.save_pretrained(adapter)\n    del trainer, model\n    gc.collect()\n    torch.cuda.empty_cache()\n    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision, quantization_config=quantization,\n                                                dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")\n    model = PeftModel.from_pretrained(model, str(adapter), is_trainable=False)\n    tokenizer = AutoTokenizer.from_pretrained(adapter)\n    tuned_summary, tuned_predictions = score(model, tokenizer, validation, test, args.output / "finetuned", args.seed)\n    comparison = make_comparison(base_summary, base_predictions, tuned_summary, tuned_predictions, args.smoke_test)\n    write_json(args.output / "comparison.json", comparison)\n    print(f"Completed. Reloaded adapter and results: {args.output}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
# Embedded from mcq_alignment.py
p = Path('mcq_alignment.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Simplified QA-conditioned representation alignment, not a KEFT reproduction."""\n\nimport csv\nfrom pathlib import Path\nimport random\n\nfrom benchmark import build_prompt\n\n\ndef load_original_qa(path):\n    # newline=\'\' preserves embedded CR/LF; never strip, condense, or normalize Q/A.\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not {"class", "Q", "A"}.issubset(reader.fieldnames or []):\n            raise ValueError("Original QA requires class,Q,A columns")\n        rows = list(reader)\n    if not rows or any(not r["Q"] or not r["A"] for r in rows):\n        raise ValueError("Empty original Q or A")\n    return rows\n\n\ndef reference_prompt(row, original_qa):\n    source = int(row["source_row"])\n    if not 0 <= source < len(original_qa):\n        raise ValueError(f"source_row {source} is outside original QA")\n    qa = original_qa[source]\n    if qa["class"] != row["class"]:\n        raise ValueError(f"source_row {source} class does not match original QA")\n    return "Original QA (project context):\\nQ:\\n" + qa["Q"] + "\\nA:\\n" + qa["A"] + "\\n\\n" + build_prompt(row)\n\n\ndef reference_ids(row, original_qa, tokenizer, max_reference_length):\n    ids = tokenizer.apply_chat_template(\n        [{"role": "user", "content": reference_prompt(row, original_qa)}],\n        tokenize=True, add_generation_prompt=True, return_dict=False,\n    )\n    if len(ids) > max_reference_length:\n        raise ValueError(f"Reference source_row={row[\'source_row\']} uses {len(ids)} tokens, above reference limit {max_reference_length}. No truncation or extra exclusion is allowed.")\n    return ids\n\n\ndef attach_references(prepared, rows, original_qa, tokenizer, max_reference_length, seed):\n    from finetune_qwen_mcq import rotate_question, tokenize_mcq\n    variants = [variant for row in rows for variant, _ in rotate_question(row, seed)]\n    random.Random(seed).shuffle(variants)  # Same deterministic order as prepare_split.\n    if len(variants) != len(prepared):\n        raise ValueError("Training presentation mismatch")\n    lengths = []\n    for example, row in zip(prepared, variants):\n        check, _ = tokenize_mcq(row, tokenizer, len(example["input_ids"]))\n        if check != example:\n            raise ValueError("Reference/student rotation mismatch")\n        ids = reference_ids(row, original_qa, tokenizer, max_reference_length)\n        example["reference_input_ids"] = ids\n        lengths.append({"source_row": int(row["source_row"]), "reference_tokens": len(ids), "student_tokens": len(example["input_ids"])})\n    return lengths\n\n\ndef alignment_smoke_rows(rows, original_qa, tokenizer, max_length, max_reference_length, seed):\n    from finetune_qwen_mcq import prepare_split, rotate_question\n    _, admitted, maps, _ = prepare_split(rows, tokenizer, max_length, seed, False)\n    longest_student = max(maps, key=lambda m: m["tokens"])["source_row"]\n    longest_reference = max(admitted, key=lambda r: max(\n        len(reference_ids(v, original_qa, tokenizer, max_reference_length))\n        for v, _ in rotate_question(r, seed)))\n    chosen = {longest_student, int(longest_reference["source_row"])}\n    selected = [r for r in admitted if int(r["source_row"]) in chosen]\n    return selected + [r for r in admitted if int(r["source_row"]) not in chosen][:8-len(selected)]\n\n\ndef collate_alignment(examples, pad_token_id):\n    import torch\n    from finetune_qwen_coder import collate_examples\n    batch = collate_examples(examples, pad_token_id)\n    if not any("reference_input_ids" in x for x in examples):\n        return batch  # Validation stays the original MCQ-only LM loss.\n    if not all("reference_input_ids" in x for x in examples):\n        raise ValueError("Mixed reference and non-reference training examples")\n    width = max(len(x["reference_input_ids"]) for x in examples)\n    batch["reference_input_ids"] = torch.tensor([\n        x["reference_input_ids"] + [pad_token_id] * (width-len(x["reference_input_ids"])) for x in examples])\n    batch["reference_attention_mask"] = torch.tensor([\n        [1] * len(x["reference_input_ids"]) + [0] * (width-len(x["reference_input_ids"])) for x in examples])\n    return batch\n\n\ndef last_prompt_positions(labels):\n    """The token immediately before the first supervised assistant-answer token."""\n    import torch\n    supervised = labels.ne(-100)\n    if not supervised.any(dim=1).all():\n        raise ValueError("An example has no supervised answer")\n    positions = supervised.long().argmax(dim=1) - 1\n    if (positions < 0).any():\n        raise ValueError("An example has no prompt before its answer")\n    return positions\n\n\ndef reference_vector(model, input_ids, attention_mask):\n    import torch\n    modes = [(module, module.training) for module in model.modules()]\n    try:\n        model.eval()\n        with model.disable_adapter(), torch.no_grad():\n            # Decoder only: avoid a huge prompt-by-vocabulary logits allocation.\n            out = model.get_base_model().model(\n                input_ids=input_ids, attention_mask=attention_mask, use_cache=False,\n                output_hidden_states=False, return_dict=True,\n            )\n            positions = attention_mask.sum(dim=1) - 1  # Reference contains prompt only.\n            vector = out.last_hidden_state[torch.arange(input_ids.shape[0], device=input_ids.device), positions].float().detach()\n        if vector.requires_grad or vector.grad_fn is not None:\n            raise RuntimeError("Reference vector must have no gradient graph")\n        return vector\n    finally:\n        # Restore exact per-module modes, including frozen/dropout modules.\n        for module, training in modes:\n            module.training = training\n\n\nclass AlignmentObjective:\n    def __init__(self, weight=0.05):\n        self.weight = weight\n        self.log = []  # Only Python scalars, never activation tensors.\n\n    def __call__(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n        import torch\n        from finetune_qwen_mcq import assistant_only_loss\n        clean = {k: v for k, v in inputs.items() if not k.startswith("reference_")}\n        if not model.training or "reference_input_ids" not in inputs:\n            return assistant_only_loss(model, clean, return_outputs, num_items_in_batch)\n        reference = reference_vector(model, inputs["reference_input_ids"], inputs["reference_attention_mask"])\n        positions = last_prompt_positions(clean["labels"])\n        captured = []\n\n        def capture(_module, _args, hidden):\n            captured.append(hidden[torch.arange(hidden.shape[0], device=hidden.device), positions].float())\n\n        # Qwen\'s final norm is after the last decoder layer, before the LM head.\n        hook = model.get_base_model().model.norm.register_forward_hook(capture)\n        try:\n            lm_loss, outputs = assistant_only_loss(model, clean, True, num_items_in_batch)\n        finally:\n            hook.remove()\n        if len(captured) != 1 or not captured[0].requires_grad:\n            raise RuntimeError("Student vector was not captured with gradients")\n        alignment_loss = (1 - torch.nn.functional.cosine_similarity(captured[0], reference, dim=-1)).mean()\n        # Trainer supplies a token denominator over the accumulation window.\n        # Scale BOTH terms consistently so 0.05 does not become 0.05 * 16.\n        scale = 1.0\n        if num_items_in_batch is not None:\n            scale = clean["labels"][:, 1:].ne(-100).sum().to(lm_loss.device) / num_items_in_batch\n        total = lm_loss + self.weight * alignment_loss * scale\n        if not all(torch.isfinite(x).all() for x in [reference, captured[0], lm_loss, alignment_loss, total]):\n            raise FloatingPointError("Non-finite reference, student vector, or loss")\n        self.log.append({"microstep": len(self.log) + 1, "lm_loss": float((lm_loss / scale).detach()),\n                         "alignment_loss": float(alignment_loss.detach()),\n                         "total_loss": float((total / scale).detach()),\n                         "reference_requires_grad": reference.requires_grad,\n                         "reference_grad_fn_is_none": reference.grad_fn is None})\n        return (total, outputs) if return_outputs else total\n', encoding='utf-8')


In [ ]:
# Embedded from compare_alignment_control.py
p = Path('compare_alignment_control.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Compare identical validation cohorts; never import unverified accuracy numbers."""\n\nimport argparse\nimport json\nfrom pathlib import Path\n\n\ndef read_result(path):\n    path = Path(path)\n    if path.is_file():\n        return json.loads(path.read_text(encoding="utf-8"))\n    import csv\n    metadata = json.loads((path / "metadata.json").read_text(encoding="utf-8"))\n    summary = json.loads((path / "finetuned/validation/summary.json").read_text(encoding="utf-8"))\n    with (path / "finetuned/validation/predictions.csv").open(encoding="utf-8-sig", newline="") as handle:\n        predictions = list(csv.DictReader(handle))\n    return {"metadata": metadata, "validation": summary, "predictions": predictions}\n\n\ndef compare(control, experiment):\n    a, b = control["metadata"], experiment["metadata"]\n    if a.get("alignment") is not None or not b.get("alignment"):\n        raise ValueError("Need a control without alignment and an experiment with alignment")\n    for key in ["model", "model_commit", "seed", "smoke_test", "admitted_questions",\n                "training_presentations", "train_label_counts", "input_sha256", "test_sha256", "settings"]:\n        if key not in a or key not in b or a[key] != b[key]:\n            raise ValueError(f"Control/experiment mismatch: {key}")\n    if sorted(a["training_source_rows"]) != sorted(b["training_source_rows"]):\n        raise ValueError("Training source cohorts differ")\n    pa, pb = control["predictions"], experiment["predictions"]\n    if len(pa) != len(pb) or not pa:\n        raise ValueError("Validation prediction cohorts differ")\n    for x, y in zip(pa, pb):\n        if any(str(x[k]) != str(y[k]) for k in ["row", "class", "question", "truth"]):\n            raise ValueError("Validation question order/content or answer key differs")\n    def correct(row):\n        return str(row["prediction"]).strip().lower() == str(row["truth"]).strip().lower()\n    ca, cb = sum(map(correct, pa)), sum(map(correct, pb))\n    for data, count in [(control, ca), (experiment, cb)]:\n        summary = data["validation"]\n        if summary["correct"] != count or summary["questions"] != len(pa) or abs(summary["accuracy"] - count/len(pa)) > 1e-10:\n            raise ValueError("Stored validation summary does not match raw predictions")\n    versions_differ = {k: {"control": a.get("versions", {}).get(k), "experiment": b.get("versions", {}).get(k)}\n                       for k in set(a.get("versions", {})) | set(b.get("versions", {}))\n                       if a.get("versions", {}).get(k) != b.get("versions", {}).get(k)}\n    return {"questions": len(pa), "control_correct": ca, "alignment_correct": cb,\n            "control_accuracy": ca/len(pa), "alignment_accuracy": cb/len(pa),\n            "accuracy_change_percentage_points": 100*(cb-ca)/len(pa),\n            "relative_accuracy_change": (cb-ca)/ca if ca else None,\n            "wrong_to_correct": sum(not correct(x) and correct(y) for x, y in zip(pa, pb)),\n            "correct_to_wrong": sum(correct(x) and not correct(y) for x, y in zip(pa, pb)),\n            "dependency_version_differences": versions_differ,\n            "same_gpu_model": a.get("gpu") == b.get("gpu"),\n            "note": "Single-seed validation comparison. Saved historical control is not a fresh paired rerun; inspect environment differences. No result establishes a KEFT reproduction or causal generalization benefit."}\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--control", type=Path, required=True)\n    parser.add_argument("--experiment", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    result = compare(read_result(args.control), read_result(args.experiment))\n    args.output.write_text(json.dumps(result, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(result, indent=2))\n', encoding='utf-8')


In [ ]:
# Embedded from tests/__init__.py
p = Path('tests/__init__.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Repository test package."""\n', encoding='utf-8')


In [ ]:
# Embedded from tests/test_mcq_alignment.py
p = Path('tests/test_mcq_alignment.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Verbatim source mapping, real tiny-Qwen hidden states, gradients and accumulation."""\nimport copy\nimport csv\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\n\nfrom mcq_alignment import (AlignmentObjective, attach_references, collate_alignment,\n                           last_prompt_positions, load_original_qa, reference_prompt,\n                           reference_vector)\nfrom finetune_qwen_mcq import assistant_only_loss, prepare_split\nfrom tests.test_mcq_rotation import ChatTokenizer, example\n\n\nclass AlignmentDataTests(unittest.TestCase):\n    def test_verbatim_qa_and_exact_rotated_student_mapping(self):\n        qa = [{"class": "Counter.cs", "Q": "  question\\r\\nwith whitespace \\t", "A": "```csharp\\r\\n  return x;\\r\\n```  "}]\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory) / "original.csv"\n            with path.open("w", encoding="utf-8", newline="") as f:\n                writer = csv.DictWriter(f, fieldnames=["class", "Q", "A"])\n                writer.writeheader()\n                writer.writerows(qa)\n            self.assertEqual(load_original_qa(path), qa)\n        row = example(0)\n        prompt = reference_prompt(row, qa)\n        self.assertIn("Q:\\n" + qa[0]["Q"] + "\\nA:\\n" + qa[0]["A"] + "\\n\\n", prompt)\n        tokenizer = ChatTokenizer()\n        prepared, admitted, _, _ = prepare_split([row], tokenizer, 10000, 42, True)\n        before = copy.deepcopy(prepared)\n        lengths = attach_references(prepared, admitted, qa, tokenizer, 20000, 42)\n        for item, original in zip(prepared, before):\n            self.assertEqual({k: v for k, v in item.items() if not k.startswith("reference_")}, original)\n            prefix = \'\'.join(map(chr, item[\'input_ids\'][:next(i for i, t in enumerate(item[\'labels\']) if t != -100)]))\n            ref = \'\'.join(map(chr, item[\'reference_input_ids\']))\n            self.assertIn(prefix.removeprefix(\'<user>\'), ref)\n        self.assertEqual(len(lengths), 4)\n        with self.assertRaisesRegex(ValueError, "No truncation"):\n            attach_references(copy.deepcopy(before), admitted, qa, tokenizer, 10, 42)\n        with self.assertRaisesRegex(ValueError, "outside"):\n            reference_prompt(example(1), qa)\n        with self.assertRaisesRegex(ValueError, "class does not match"):\n            reference_prompt({**row, "class": "Other.cs"}, qa)\n\n\nclass AlignmentGradientTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        import torch\n        from transformers import Qwen2Config, Qwen2ForCausalLM\n        from peft import LoraConfig, get_peft_model\n        cls.t = torch\n        torch.set_num_threads(2)\n        torch.manual_seed(123)\n        base = Qwen2ForCausalLM(Qwen2Config(\n            vocab_size=101, hidden_size=32, intermediate_size=64,\n            num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2,\n            max_position_embeddings=128, attention_dropout=0.0,\n        ))\n        cls.model = get_peft_model(base, LoraConfig(r=4, lora_alpha=8, lora_dropout=0,\n                                  target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))\n        # Nonzero adapters make teacher isolation meaningful, even before an update.\n        with torch.no_grad():\n            for name, parameter in cls.model.named_parameters():\n                if "lora_B" in name:\n                    parameter.normal_(std=0.04)\n        ids = torch.randint(0, 101, (2, 18))\n        labels = ids.clone()\n        labels[:, :14] = -100  # Last prompt token is 13, never answer token 14.\n        ref = torch.randint(0, 101, (2, 26))\n        cls.batch = {"input_ids": ids, "labels": labels, "attention_mask": torch.ones_like(ids),\n                     "reference_input_ids": ref, "reference_attention_mask": torch.ones_like(ref)}\n\n    def setUp(self):\n        self.model.zero_grad(set_to_none=True)\n        self.model.train()\n\n    def test_reference_is_base_eval_detached_and_restores_modes_and_adapters(self):\n        t, model, b = self.t, self.model, self.batch\n        before = [m.training for m in model.modules()]\n        observed = []\n        def check_mode(module, _inputs):\n            observed.append((module.training, t.is_grad_enabled()))\n        handle = model.get_base_model().model.register_forward_pre_hook(check_mode)\n        try:\n            ref = reference_vector(model, b[\'reference_input_ids\'], b[\'reference_attention_mask\'])\n        finally:\n            handle.remove()\n        self.assertEqual(observed, [(False, False)])\n        self.assertFalse(ref.requires_grad)\n        self.assertIsNone(ref.grad_fn)\n        self.assertEqual(before, [m.training for m in model.modules()])\n        self.assertTrue(all(not m.disable_adapters for m in model.modules() if hasattr(m, \'disable_adapters\') and isinstance(m.disable_adapters, bool)))\n        with t.no_grad():\n            for n, p in model.named_parameters():\n                if \'lora_B\' in n:\n                    p.add_(0.01)\n        other = reference_vector(model, b[\'reference_input_ids\'], b[\'reference_attention_mask\'])\n        t.testing.assert_close(ref, other, rtol=0, atol=0)\n        self.assertTrue(all(p.grad is None for p in model.parameters()))\n\n    def test_total_and_last_prompt_vectors_match_explicit_hidden_state_formula(self):\n        t, model, b = self.t, self.model, self.batch\n        objective = AlignmentObjective(.05)\n        total = objective(model, b)\n        clean = {k: v for k, v in b.items() if not k.startswith(\'reference_\')}\n        lm = assistant_only_loss(model, clean)\n        out = model.get_base_model().model(input_ids=b[\'input_ids\'], attention_mask=b[\'attention_mask\'], output_hidden_states=True, use_cache=False)\n        positions = last_prompt_positions(b[\'labels\'])\n        self.assertEqual(positions.tolist(), [13, 13])\n        student = out.hidden_states[-1][t.arange(2), positions]\n        ref = reference_vector(model, b[\'reference_input_ids\'], b[\'reference_attention_mask\'])\n        alignment = (1-t.nn.functional.cosine_similarity(student, ref, dim=-1)).mean()\n        t.testing.assert_close(total, lm + .05 * alignment)\n        # Alignment alone must reach LoRA, not just the supervised LM term.\n        alignment.backward()\n        grads = [p.grad for n, p in model.named_parameters() if \'lora_\' in n and p.grad is not None]\n        self.assertTrue(grads and any(t.count_nonzero(g).item() for g in grads))\n        self.assertTrue(all(t.isfinite(g).all() for g in grads))\n        self.assertTrue(all(p.grad is None for n, p in model.named_parameters() if \'lora_\' not in n))\n        altered = {**b, \'input_ids\': b[\'input_ids\'].clone()}\n        altered[\'input_ids\'][:, 14:] = (altered[\'input_ids\'][:, 14:] + 1) % 101\n        other = AlignmentObjective(.05)\n        other(model, altered)\n        self.assertAlmostEqual(objective.log[0][\'alignment_loss\'], other.log[0][\'alignment_loss\'], places=6)\n\n    def test_accumulation_matches_whole_batch_and_checkpointed_backward(self):\n        t, model, b = self.t, self.model, self.batch\n        total = AlignmentObjective()(model, b)\n        total.backward()\n        expected = {n: p.grad.clone() for n, p in model.named_parameters() if p.grad is not None}\n        model.zero_grad(set_to_none=True)\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\': False})\n        try:\n            denominator = b[\'labels\'][:, 1:].ne(-100).sum()\n            for i in range(2):\n                micro = {k: v[i:i+1] for k, v in b.items()}\n                loss = AlignmentObjective()(model, micro, num_items_in_batch=denominator)\n                self.assertTrue(t.isfinite(loss))\n                loss.backward()\n            for n, p in model.named_parameters():\n                if n in expected:\n                    t.testing.assert_close(p.grad, expected[n], rtol=5e-4, atol=2e-6)\n        finally:\n            model.gradient_checkpointing_disable()\n\n    def test_eval_uses_unchanged_lm_loss_without_reference_forward(self):\n        self.model.eval()\n        objective = AlignmentObjective()\n        with patch(\'mcq_alignment.reference_vector\', side_effect=AssertionError(\'Reference used in validation\')):\n            loss = objective(self.model, self.batch)\n        clean = {k: v for k, v in self.batch.items() if not k.startswith(\'reference_\')}\n        self.t.testing.assert_close(loss, assistant_only_loss(self.model, clean))\n        self.assertEqual(objective.log, [])\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', encoding='utf-8')


In [ ]:
# Embedded from tests/test_mcq_rotation.py
p = Path('tests/test_mcq_rotation.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('import copy\nimport unittest\nfrom collections import Counter\n\nfrom finetune_qwen_mcq import check_split_isolation, prepare_split, rotate_question, rotation_diagnostics, tokenize_mcq, smoke_training_rows\n\n\nclass ChatTokenizer:\n    """Character tokenizer with an explicit assistant prefix and end token."""\n    def apply_chat_template(self, messages, tokenize, add_generation_prompt, return_dict=True):\n        text = "".join(f"<{m[\'role\']}>" + m[\'content\'] + "</turn>" for m in messages)\n        if add_generation_prompt:\n            text += "<assistant>"\n        ids = [ord(c) for c in text]\n        return {"input_ids": ids} if return_dict else ids\n\n\ndef example(source_id=1):\n    return {"source_row": str(source_id), "split": "train", "class": "Counter.cs",\n            "Question": f"Which statement is used in operation {source_id}?",\n            "a": "```csharp\\nCounter.instance\\n```", "b": "```csharp\\nCounter.Instance\\n```",\n            "c": \'return "a  b";\', "d": \'return "a b";\', "Answer": "b"}\n\n\nclass RotationTests(unittest.TestCase):\n    def test_every_option_visits_every_position_and_target_stays_correct(self):\n        for correct in "abcd":\n            row = {**example(), "Answer": correct}\n            before = copy.deepcopy(row)\n            variants = rotate_question(row, 42)\n            self.assertEqual(row, before)\n            self.assertEqual(Counter(v[\'Answer\'] for v, _ in variants), dict.fromkeys(\'abcd\', 1))\n            for letter in "abcd":\n                self.assertEqual({v[letter] for v, _ in variants}, {row[l] for l in \'abcd\'})\n            for v, mapping in variants:\n                self.assertEqual(v[v[\'Answer\']], row[correct])\n                self.assertEqual(v[\'Question\'], row[\'Question\'])\n                self.assertEqual(v[\'source_row\'], row[\'source_row\'])\n                for new, original in mapping[\'new_to_original\'].items():\n                    self.assertEqual(v[new], row[original])\n\n    def test_rotation_is_reproducible_independent_of_input_order(self):\n        rows = [example(i) for i in range(10)]\n        forward = {r[\'source_row\']: rotate_question(r, 7) for r in rows}\n        reverse = {r[\'source_row\']: rotate_question(r, 7) for r in reversed(rows)}\n        self.assertEqual(forward, reverse)\n        self.assertNotEqual(forward, {r[\'source_row\']: rotate_question(r, 8) for r in rows})\n\n    def test_smoke_includes_longest_fit_excludes_overlength_and_keeps_unique_ids(self):\n        rows = [example(i) for i in range(12)]\n        rows[10][\'Question\'] += \' context\' * 400\n        rows[11][\'Question\'] += \' context\' * 2000\n        selected = smoke_training_rows(rows, ChatTokenizer(), 10000, 42)\n        self.assertEqual(len(selected), 8)\n        self.assertEqual(selected[0][\'source_row\'], \'10\')\n        self.assertNotIn(\'11\', [r[\'source_row\'] for r in selected])\n        self.assertEqual(len({r[\'source_row\'] for r in selected}), 8)\n\n    def test_only_exact_assistant_answer_and_turn_end_receive_loss(self):\n        row = example()\n        item, _ = tokenize_mcq(row, ChatTokenizer(), 10000)\n        supervised = [token for token in item[\'labels\'] if token != -100]\n        self.assertEqual(\'\'.join(map(chr, supervised)), \'Answer:b</turn>\')\n        self.assertEqual(item[\'labels\'][-len(supervised):], supervised)\n        self.assertTrue(all(label == -100 for label in item[\'labels\'][:-len(supervised)]))\n        self.assertEqual(len(item[\'input_ids\']), len(item[\'labels\']))\n\n    def test_bad_template_prefix_fails_loudly(self):\n        class BadTokenizer(ChatTokenizer):\n            def apply_chat_template(self, messages, tokenize, add_generation_prompt, return_dict=True):\n                out = super().apply_chat_template(messages, tokenize, add_generation_prompt, return_dict=return_dict)\n                return [999] + out if not add_generation_prompt else out\n        with self.assertRaisesRegex(ValueError, \'prefix mismatch\'):\n            tokenize_mcq(example(), BadTokenizer(), 10000)\n\n    def test_group_length_rejection_preserves_fourfold_balance(self):\n        class OrderLengthTokenizer(ChatTokenizer):\n            def apply_chat_template(self, messages, tokenize, add_generation_prompt, return_dict=True):\n                out = super().apply_chat_template(messages, tokenize, add_generation_prompt, return_dict=return_dict)\n                # Exactly one order of source 2 becomes overlength.\n                if \'operation 2?\' in messages[0][\'content\'] and \'a) ```csharp\\nCounter.instance\' in messages[0][\'content\']:\n                    out = [1] * 20000 + out\n                return out\n        prepared, admitted, maps, exclusions = prepare_split([example(1), example(2)], OrderLengthTokenizer(), 10000, 42, True)\n        self.assertEqual(len(prepared), 4)\n        self.assertEqual([r[\'source_row\'] for r in admitted], [\'1\'])\n        self.assertEqual([r[\'source_row\'] for r in exclusions], [2])\n        self.assertEqual(Counter(m[\'answer\'] for m in maps), dict.fromkeys(\'abcd\', 1))\n\n    def test_fixed_validation_and_no_split_overlap(self):\n        row = {**example(2), \'split\': \'validation\'}\n        check_split_isolation([example()], [row])\n        prepared, admitted, maps, exclusions = prepare_split([row], ChatTokenizer(), 10000, 42, False)\n        expected, _ = tokenize_mcq(row, ChatTokenizer(), 10000)\n        self.assertEqual(prepared, [expected])\n        self.assertEqual(admitted, [row])\n        self.assertEqual(len(maps), 4)\n        self.assertFalse(exclusions)\n        with self.assertRaisesRegex(ValueError, \'Source ids overlap\'):\n            check_split_isolation([example()], [{**row, \'source_row\': \'1\'}])\n        with self.assertRaisesRegex(ValueError, \'questions cross splits\'):\n            check_split_isolation([example()], [{**row, \'Question\': example()[\'Question\'].upper()}])\n\n    def test_diagnostic_maps_predictions_back_to_original_answer_text(self):\n        rows = [example(1), example(2), example(3)]\n        predictions = []\n        for index, row in enumerate(rows):\n            for variant, mapping in rotate_question(row, 42):\n                # First always correct, second always same wrong text, third always invalid.\n                original = row[\'Answer\'] if index == 0 else \'a\'\n                predicted = next(l for l, old in mapping[\'new_to_original\'].items() if old == original) if index < 2 else None\n                predictions.append({\'prediction\': predicted, \'correct\': predicted == variant[\'Answer\']})\n        out = rotation_diagnostics(rows, predictions, 42)\n        self.assertAlmostEqual(out[\'consistent_answer_text_rate\'], 2 / 3)\n        self.assertAlmostEqual(out[\'all_four_correct_rate\'], 1 / 3)\n        for stats in out[\'accuracy_by_correct_position\'].values():\n            self.assertEqual(stats[\'questions\'], 3)\n            self.assertEqual(stats[\'correct\'], 1)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', encoding='utf-8')


In [ ]:
# Embedded from tests/test_mcq_loss.py
p = Path('tests/test_mcq_loss.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Check loss/gradient equivalence on a real, randomly initialized tiny Qwen."""\n\nimport unittest\n\n\nclass MCQLossTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        import torch\n        from transformers import Qwen2Config, Qwen2ForCausalLM\n        cls.torch = torch\n        torch.set_num_threads(2)\n        torch.manual_seed(42)\n        cls.model = Qwen2ForCausalLM(Qwen2Config(\n            vocab_size=101, hidden_size=32, intermediate_size=64,\n            num_hidden_layers=1, num_attention_heads=4, num_key_value_heads=2,\n            max_position_embeddings=128, attention_dropout=0.0,\n        )).eval()\n\n    def test_loss_and_all_parameter_gradients_match_dense_masked_causal_loss(self):\n        from finetune_qwen_mcq import assistant_only_loss\n        t = self.torch\n        ids = t.randint(0, 101, (2, 24))\n        labels = ids.clone()\n        labels[0, :18] = -100\n        labels[1, :15] = -100\n        labels[1, 21:] = -100\n        mask = t.ones_like(ids)\n        mask[1, 21:] = 0\n        batch = {"input_ids": ids, "attention_mask": mask, "labels": labels}\n        for denominator in [None, t.tensor(24)]:\n            self.model.zero_grad(set_to_none=True)\n            output = self.model(input_ids=ids, attention_mask=mask, use_cache=False)\n            full_loss = t.nn.functional.cross_entropy(\n                output.logits[:, :-1].reshape(-1, 101), labels[:, 1:].reshape(-1),\n                ignore_index=-100, reduction="mean" if denominator is None else "sum")\n            if denominator is not None:\n                full_loss = full_loss / denominator\n            full_loss.backward()\n            gradients = {n: p.grad.clone() for n, p in self.model.named_parameters() if p.grad is not None}\n            self.model.zero_grad(set_to_none=True)\n            small_loss, reduced = assistant_only_loss(self.model, batch, True, denominator)\n            t.testing.assert_close(small_loss, full_loss)\n            small_loss.backward()\n            for name, p in self.model.named_parameters():\n                if name in gradients:\n                    t.testing.assert_close(p.grad, gradients[name], rtol=2e-4, atol=2e-6)\n            self.assertLess(reduced.logits.shape[1], ids.shape[1])\n            self.assertIn("labels", batch)  # Loss must not mutate the collator batch.\n\n    def test_no_supervised_targets_rejected(self):\n        from finetune_qwen_mcq import assistant_only_loss\n        t = self.torch\n        batch = {"input_ids": t.ones((1, 8), dtype=t.long),\n                 "labels": t.full((1, 8), -100, dtype=t.long)}\n        with self.assertRaisesRegex(ValueError, "no supervised"):\n            assistant_only_loss(self.model, batch)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')


In [ ]:
# Embedded from tests/test_mcq_baseline.py
p = Path('tests/test_mcq_baseline.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('import copy\nimport unittest\n\nfrom finetune_qwen_mcq import make_comparison\n\n\nclass BaselineTests(unittest.TestCase):\n    def test_skipped_baseline_has_no_invented_metrics(self):\n        tuned = {"test": {"accuracy": 0.75}}\n        result = make_comparison(None, None, tuned, [{"correct": True}])\n        self.assertIsNone(result["base"])\n        self.assertEqual(result["baseline_evaluation"], "skipped")\n        for key in ["test_accuracy_change_percentage_points", "wrong_to_correct", "correct_to_wrong"]:\n            self.assertIsNone(result[key])\n        self.assertEqual(result["finetuned"], tuned)\n\n    def test_measured_baseline_retains_delta_and_prediction_flips(self):\n        base, tuned = {"test": {"accuracy": 0.5}}, {"test": {"accuracy": 0.75}}\n        before = copy.deepcopy((base, tuned))\n        old = [{"correct": x} for x in [False, True, False, True]]\n        new = [{"correct": x} for x in [True, False, True, True]]\n        result = make_comparison(base, old, tuned, new, True)\n        self.assertEqual(result["test_accuracy_change_percentage_points"], 25)\n        self.assertEqual(result["wrong_to_correct"], 2)\n        self.assertEqual(result["correct_to_wrong"], 1)\n        self.assertEqual(result["baseline_evaluation"], "measured_this_run")\n        self.assertTrue(result["smoke_test"])\n        self.assertEqual((base, tuned), before)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')


In [ ]:
# Embedded from tests/test_alignment_comparison.py
p = Path('tests/test_alignment_comparison.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Guard against mismatched cohorts, fabricated summaries, and hidden environment changes."""\nimport copy\nimport unittest\n\nfrom compare_alignment_control import compare\n\n\nclass AlignmentComparisonTests(unittest.TestCase):\n    def pair(self):\n        metadata = {"model": "tiny", "model_commit": "revision", "seed": 42, "smoke_test": False,\n                    "admitted_questions": {"train": 2, "validation": 2}, "training_presentations": 8,\n                    "train_label_counts": dict.fromkeys("abcd", 2), "input_sha256": {"train": "t", "validation": "v"},\n                    "test_sha256": "test", "settings": {"max_length": 4096}, "training_source_rows": [1, 2],\n                    "gpu": "T4", "versions": {"torch": "a"}}\n        predictions = [{"row": str(i), "class": "X", "question": str(i), "truth": "a", "prediction": p}\n                       for i, p in enumerate(["a", "b"])]\n        control = {"metadata": metadata, "validation": {"correct": 1, "questions": 2, "accuracy": .5}, "predictions": predictions}\n        experiment = copy.deepcopy(control)\n        experiment["metadata"]["alignment"] = {"weight": .05}\n        experiment["predictions"][1]["prediction"] = "a"\n        experiment["validation"] = {"correct": 2, "questions": 2, "accuracy": 1.0}\n        return control, experiment\n\n    def test_delta_and_paired_changes_use_verified_predictions(self):\n        control, experiment = self.pair()\n        result = compare(control, experiment)\n        self.assertEqual(result["accuracy_change_percentage_points"], 50)\n        self.assertEqual(result["relative_accuracy_change"], 1)\n        self.assertEqual(result["wrong_to_correct"], 1)\n        self.assertEqual(result["correct_to_wrong"], 0)\n\n    def test_mismatched_settings_cohort_and_summary_are_rejected(self):\n        for change in ["settings", "question", "summary"]:\n            control, experiment = self.pair()\n            if change == "settings":\n                experiment["metadata"]["settings"]["max_length"] = 2048\n            elif change == "question":\n                experiment["predictions"][0]["question"] = "different"\n            else:\n                experiment["validation"]["correct"] = 1\n            with self.assertRaises(ValueError):\n                compare(control, experiment)\n\n    def test_environment_difference_is_disclosed(self):\n        control, experiment = self.pair()\n        experiment["metadata"]["versions"]["torch"] = "b"\n        self.assertEqual(compare(control, experiment)["dependency_version_differences"]["torch"], {"control": "a", "experiment": "b"})\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')


In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_mcq_alignment',
                'tests.test_mcq_loss', 'tests.test_mcq_rotation', 'tests.test_mcq_baseline', 'tests.test_alignment_comparison', '-v'], check=True)


## Preflight: exact cohorts and longer reference lengths

No model is loaded or trained here. Expected full admission is 1,717 training QA × four rotations
= 6,868 presentations and 192 validation questions, with the same single overlength student exclusion.
The original QA CSV has 1,910 rows. All additional reference prompts are logged without truncation.


In [ ]:
def archive_results():
    archive = shutil.make_archive(str(EXPERIMENT_DIR), 'zip', str(EXPERIMENT_DIR))
    print('Saved output archive:', archive, flush=True)
    return archive

for name in ['benchmark.py', 'finetune_qwen_coder.py', 'finetune_qwen_mcq.py', 'mcq_alignment.py', 'compare_alignment_control.py']:
    shutil.copyfile(name, EXPERIMENT_DIR / name)
shutil.copyfile('control_results.json', EXPERIMENT_DIR / 'historical_control.json')
(EXPERIMENT_DIR / 'input_hashes.json').write_text(json.dumps(EXPECTED, indent=2))

def run_experiment(folder, alignment=True, *flags):
    command = [sys.executable, '-u', 'finetune_qwen_mcq.py',
               '--train', 'train.csv', '--validation', 'validation.csv', '--test', 'TestData.csv',
               '--output', str(folder), '--max-length', str(MAX_LENGTH), '--seed', str(SEED), '--revision', MODEL_REVISION,
               '--skip-base-evaluation']
    if alignment:
        command += ['--alignment-qa', 'TrainData.csv', '--max-reference-length', str(MAX_REFERENCE_LENGTH)]
    command += list(flags)
    log_path = EXPERIMENT_DIR / (Path(folder).name + '-console.log')
    try:
        with log_path.open('w', encoding='utf-8') as log:
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        if code:
            raise RuntimeError(f'Run failed ({code}); inspect {log_path}. Partial results are archived; no silent fallback or truncation.')
    finally:
        archive_results()

run_experiment(EXPERIMENT_DIR / 'preflight', True, '--prepare-only')
preflight = json.loads((EXPERIMENT_DIR / 'preflight/metadata.json').read_text())
control = json.loads(Path('control_results.json').read_text())
for key in ['admitted_questions', 'training_presentations', 'training_source_rows', 'input_sha256', 'train_label_counts', 'settings']:
    assert preflight[key] == control['metadata'][key], f'Cohort/settings mismatch: {key}'
print('Student admission:', preflight['admitted_questions'])
print('Maximum reference tokens:', preflight['alignment']['max_observed_reference_tokens'])


## GPU smoke test

Checks finite LM/alignment/total losses, detached reference vectors, finite nonzero LoRA gradients,
no base gradients, and adapter save/reload. Subset accuracy is not a benchmark result.


In [ ]:
import math
if RUN_SMOKE:
    folder = EXPERIMENT_DIR / 'smoke'
    run_experiment(folder, True, '--smoke-test')
    checks = json.loads((folder / 'alignment_gradient_check.json').read_text())
    losses = json.loads((folder / 'alignment_loss_log.json').read_text())
    assert checks['lora_gradients_present_finite_nonzero'] and checks['frozen_base_has_no_gradients']
    assert checks['reference_has_no_gradients'] and losses
    assert all(math.isfinite(x[k]) for x in losses for k in ['lm_loss', 'alignment_loss', 'total_loss'])
    assert all(not x['reference_requires_grad'] and x['reference_grad_fn_is_none'] for x in losses)
    assert json.loads((folder / 'comparison.json').read_text())['saved_adapter_reload_verified']
    print('GPU smoke passed. Full training below starts with a NEW adapter.')


## Full one-epoch experiment

The optional control and alignment experiment each start from pretrained weights with fresh adapters.
Only the alignment run receives original QA in its training reference pass. All generated evaluation
prompts remain MCQ-only, and saving/reloading and scoring use the existing pipeline.


In [ ]:
CONTROL_SOURCE = Path('control_results.json')
if RUN_MATCHED_CONTROL:
    CONTROL_SOURCE = EXPERIMENT_DIR / 'control'
    run_experiment(CONTROL_SOURCE, False)
run_experiment(ALIGNMENT_DIR, True)


## Compare validation MCQ accuracy with the control

The comparison verifies identical data hashes, seed, hyperparameters, admitted source rows,
validation questions/order/keys, and raw predictions against stored counts. Dependency differences
are reported. A saved historical control is not a freshly paired run; one seed cannot establish generalization.


In [ ]:
from compare_alignment_control import compare, read_result
from IPython.display import Markdown, display
comparison = compare(read_result(CONTROL_SOURCE), read_result(ALIGNMENT_DIR))
comparison['control_origin'] = 'fresh_same_notebook_run' if RUN_MATCHED_CONTROL else 'saved_historical_run'
(EXPERIMENT_DIR / 'validation_comparison.json').write_text(json.dumps(comparison, indent=2) + '\n')
display(Markdown('| Validation metric | Control | Alignment |\n|---|---:|---:|\n'
    f"| Correct | {comparison['control_correct']}/{comparison['questions']} | {comparison['alignment_correct']}/{comparison['questions']} |\n"
    f"| Accuracy | {comparison['control_accuracy']:.2%} | {comparison['alignment_accuracy']:.2%} |"))
print('Change, percentage points:', comparison['accuracy_change_percentage_points'])
print('Relative accuracy change:', comparison['relative_accuracy_change'])
print('Wrong → correct:', comparison['wrong_to_correct'], '; correct → wrong:', comparison['correct_to_wrong'])
print('Environment differences:', comparison['dependency_version_differences'])
print('Control origin:', comparison['control_origin'])
archive_results()


## Download and inspect evidence

Download the ZIP from the saved version's **Output**. The alignment adapter is under
`alignment/adapter`; accuracy and raw predictions are under `alignment/finetuned`.
`alignment_gradient_check.json`, `alignment_loss_log.json`, `alignment_lengths.json`,
`metadata.json`, `training_metrics.json`, and `validation_comparison.json` record the experiment.
Reference vectors are intentionally not saved or cached. This notebook has not already trained the new experiment.


In [ ]:
from IPython.display import FileLink
archive = archive_results()
display(FileLink(str(Path(archive).relative_to(WORKING_DIR))))
